# Build researcher reports for saved Cisco runs

**MSR 2027 Mining Challenge · GitSkills team**

This notebook turns saved Cisco Skill Scanner runs into reports a researcher can navigate. It **never runs Cisco**
and **only reads** the run folders (`results/cisco_dataset_runs/`), the sample folders and the manual-review
worksheets. It writes only to `results/reports/`.

For every run (and every notebook-02 combined report) it writes `results/reports/<database>/<run ID>/`:

| folder | open it for |
|---|---|
| `01_IMPORTANT_RESULTS/` | **`START_HERE.md`** and **`dashboard.html`**, plus coverage, key-findings, severity and comparison tables |
| `02_DETAILED_ANALYSIS/` | one page per skill with security alerts, rule / analyzer breakdowns, the full finding tables, figures |
| `03_RAW_SCANNER_OUTPUT/` | where the raw Cisco JSON is, with SHA-256 per report (nothing large is copied), metadata copies |
| `04_HUMAN_VALIDATION/` | what has been validated, and snapshots of the manual-review worksheet |

`results/reports/index.html` and `README.md` list every report; ★ marks the one to open first for each experiment.

### Scanner alerts vs validated findings

Every table separates three tiers: **scanner alert** (Cisco reported it), **provisional** (a researcher-approved,
AI-assisted label, not an independent human review) and **human-validated** (`final_label` and `adjudicated_by`
filled in the worksheet). Only the last tier may be reported as a validated finding.

### Safety

Skill content is untrusted. Every value that reaches a report is redacted where it looks like a secret
(`[REDACTED]`), absolute paths become repository-relative, the dashboard renders untrusted text as plain text
under a strict Content-Security-Policy, and each finished report is audited for secrets and local paths before it
replaces an earlier build. A report folder is only ever replaced by a newer build of itself.

### New analyzers

Configurations come from each run's `run_metadata.json`, so LLM Analyzer and Meta-Analyzer configurations (or any
future one) appear automatically. Describe a new analyzer mode once in `ANALYZER_MODES` in
`src/gitskills_reports/model.py`.

## 1 — Settings

* `REPORTS` — `"all"` rebuilds every saved run and combined report. Otherwise give a list of run folders (or
  notebook-02 `combined/` folders), relative to the repository root.
* `RUN_TESTS` — run the validation tests in `tests/test_reporting.py` after building.

In [ ]:
REPORTS = "all"            # or ["results/cisco_dataset_runs/agent_skills_sample/runs/2026-11-03_195654_sample30-seed2027-reanalysis"]
RUN_TESTS = True

## 2 — Imports and paths

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
ROOT = next(f for f in (cwd, *cwd.parents) if (f / "notebooks").is_dir() and (f / "results").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from gitskills_reports import GENERATOR_VERSION, build_all, build_report, load_combined, load_run, refresh_index

OUT_ROOT = ROOT / "results" / "reports"
print("Repository :", ROOT)
print("Reports    :", OUT_ROOT.relative_to(ROOT), "| generator", GENERATOR_VERSION)

## 3 — Build the reports

In [ ]:
if REPORTS == "all":
    built = build_all(ROOT, OUT_ROOT, log=print)
else:
    built = []
    for folder in REPORTS:
        folder = (ROOT / folder).resolve()
        source = load_combined(folder, ROOT) if (folder / "combined_metadata.json").is_file() else load_run(folder)
        built.append(build_report(source, ROOT, OUT_ROOT))
        print("built", built[-1].relative_to(ROOT))
    refresh_index(ROOT, OUT_ROOT)
print(f"\n{len(built)} report(s). Index: {(OUT_ROOT / 'index.html').relative_to(ROOT)}")

## 4 — Where to look

The index below links every report's `START_HERE.md`. Dashboards open in any browser straight from disk (double-click
the file, or paste the `file://` link printed below into the address bar); they need no server and no network.

In [ ]:
display(Markdown((OUT_ROOT / "README.md").read_text(encoding="utf-8").replace("](", "](../results/reports/")))
for report in built:
    print(report.relative_to(ROOT).as_posix().ljust(95), (report / "01_IMPORTANT_RESULTS" / "dashboard.html").as_uri())

## 5 — Validation tests

In [ ]:
if RUN_TESTS:
    env = {**os.environ, "PYTHONPATH": str(ROOT / "src"), "PYTHONDONTWRITEBYTECODE": "1"}
    result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=ROOT, env=env,
                            capture_output=True, text=True)
    print(result.stderr[-6000:])
    assert result.returncode == 0, "validation tests failed (see above)"